In [13]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/pankrzysiu/cifar10-python/cifar-10-python.tar.gz
/kaggle/input/datasets/pankrzysiu/cifar10-python/cifar-10-batches-py/data_batch_1
/kaggle/input/datasets/pankrzysiu/cifar10-python/cifar-10-batches-py/data_batch_2
/kaggle/input/datasets/pankrzysiu/cifar10-python/cifar-10-batches-py/batches.meta
/kaggle/input/datasets/pankrzysiu/cifar10-python/cifar-10-batches-py/test_batch
/kaggle/input/datasets/pankrzysiu/cifar10-python/cifar-10-batches-py/data_batch_3
/kaggle/input/datasets/pankrzysiu/cifar10-python/cifar-10-batches-py/data_batch_5
/kaggle/input/datasets/pankrzysiu/cifar10-python/cifar-10-batches-py/data_batch_4
/kaggle/input/datasets/pankrzysiu/cifar10-python/cifar-10-batches-py/readme.html


In [14]:
import torch
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

# 1. Define Transformations
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)) # Normalizes to [-1, 1]
])

# 2. Set the EXACT path from your list (the parent folder of 'cifar-10-batches-py')
data_root = '/kaggle/input/datasets/pankrzysiu/cifar10-python/' 

# 3. Load Train and Test sets (download=False because it's already there!)
trainset = torchvision.datasets.CIFAR10(root=data_root, train=True,
                                        download=False, transform=transform)
trainloader = DataLoader(trainset, batch_size=64, shuffle=True, num_workers=2)

testset = torchvision.datasets.CIFAR10(root=data_root, train=False,
                                       download=False, transform=transform)
testloader = DataLoader(testset, batch_size=64, shuffle=False, num_workers=2)

print(f"Training set size: {len(trainset)}")
print(f"Test set size: {len(testset)}")

Training set size: 50000
Test set size: 10000


In [15]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class PatchEmbedding(nn.Module):
    def __init__(self, in_channels=3, patch_size=4, embed_dim=128, img_size=32):
        super().__init__()
        self.patch_size = patch_size
        # Conv2d with kernel=stride=patch_size acts as patch extraction + linear projection
        self.proj = nn.Conv2d(in_channels, embed_dim, kernel_size=patch_size, stride=patch_size)
        self.num_patches = (img_size // patch_size) ** 2

    def forward(self, x):
        # x: [B, 3, 32, 32] -> [B, embed_dim, 8, 8]
        x = self.proj(x)
        # Flatten spatial dimensions: [B, embed_dim, 64] -> [B, 64, embed_dim]
        x = x.flatten(2).transpose(1, 2)
        return x

class MultiHeadSelfAttention(nn.Module):
    def __init__(self, embed_dim, num_heads=4):
        super().__init__()
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.scale = self.head_dim ** -0.5
        
        self.qkv = nn.Linear(embed_dim, embed_dim * 3)
        self.proj = nn.Linear(embed_dim, embed_dim)

    def forward(self, x):
        B, N, C = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2] # [B, heads, N, head_dim]
        
        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)
        
        x = (attn @ v).transpose(1, 2).reshape(B, N, C)
        x = self.proj(x)
        return x

class TransformerBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, mlp_ratio=4.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(embed_dim)
        self.attn = MultiHeadSelfAttention(embed_dim, num_heads)
        self.norm2 = nn.LayerNorm(embed_dim)
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, int(embed_dim * mlp_ratio)),
            nn.GELU(),
            nn.Linear(int(embed_dim * mlp_ratio), embed_dim)
        )

    def forward(self, x):
        # Residual connections
        x = x + self.attn(self.norm1(x))
        x = x + self.mlp(self.norm2(x))
        return x

class ViT(nn.Module):
    def __init__(self, img_size=32, patch_size=4, in_channels=3, num_classes=10,
                 embed_dim=128, depth=6, num_heads=4, mlp_ratio=4.0):
        super().__init__()
        self.patch_embed = PatchEmbedding(in_channels, patch_size, embed_dim, img_size)
        num_patches = self.patch_embed.num_patches

        # Learnable [CLS] token
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        # Learnable Positional Embeddings
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches + 1, embed_dim))
        
        self.blocks = nn.ModuleList([
            TransformerBlock(embed_dim, num_heads, mlp_ratio) for _ in range(depth)
        ])
        self.norm = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, num_classes)

        # Initialize weights
        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            nn.init.trunc_normal_(m.weight, std=0.02)
            if m.bias is not None: nn.init.constant_(m.bias, 0)
        elif isinstance(m, nn.LayerNorm):
            nn.init.constant_(m.bias, 0)
            nn.init.constant_(m.weight, 1.0)

    def forward(self, x):
        B = x.shape[0]
        x = self.patch_embed(x) # [B, 64, 128]
        
        # Add CLS token
        cls_tokens = self.cls_token.expand(B, -1, -1)
        x = torch.cat((cls_tokens, x), dim=1) # [B, 65, 128]
        
        # Add positional embedding
        x = x + self.pos_embed
        
        # Pass through Transformer blocks
        for block in self.blocks:
            x = block(x)
        
        x = self.norm(x)
        # Extract CLS token for classification
        cls_out = x[:, 0]
        return self.head(cls_out)

# --- Instantiate the model ---
# Make sure GPU is enabled in the right sidebar!
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = ViT().to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"ViT Parameters: {total_params / 1e6:.2f}M")
print(f"Using device: {device}")

ViT Parameters: 1.21M
Using device: cuda


In [16]:
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.05)
scheduler = CosineAnnealingLR(optimizer, T_max=20) # For 20 epochs

def train_one_epoch(epoch):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    for i, (inputs, labels) in enumerate(trainloader):
        inputs, labels = inputs.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        
    train_acc = 100. * correct / total
    print(f"Epoch {epoch}: Train Loss: {running_loss/len(trainloader):.4f}, Train Acc: {train_acc:.2f}%")

def evaluate():
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, labels in testloader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
    return 100. * correct / total

# --- Run Training ---
epochs = 20 
for epoch in range(1, epochs + 1):
    train_one_epoch(epoch)
    test_acc = evaluate()
    scheduler.step()
    print(f"Epoch {epoch}: Test Accuracy: {test_acc:.2f}%\n")

print("Finished Training!")

# Save the model so you don't lose it when the session ends
torch.save(model.state_dict(), '/kaggle/working/vit_cifar10.pth')
print("Model saved to /kaggle/working/vit_cifar10.pth")

Epoch 1: Train Loss: 1.7606, Train Acc: 34.56%
Epoch 1: Test Accuracy: 43.17%

Epoch 2: Train Loss: 1.4243, Train Acc: 47.77%
Epoch 2: Test Accuracy: 51.77%

Epoch 3: Train Loss: 1.2418, Train Acc: 55.00%
Epoch 3: Test Accuracy: 55.91%

Epoch 4: Train Loss: 1.1388, Train Acc: 58.92%
Epoch 4: Test Accuracy: 59.83%

Epoch 5: Train Loss: 1.0577, Train Acc: 62.11%
Epoch 5: Test Accuracy: 61.29%

Epoch 6: Train Loss: 0.9909, Train Acc: 64.64%
Epoch 6: Test Accuracy: 62.75%

Epoch 7: Train Loss: 0.9316, Train Acc: 66.64%
Epoch 7: Test Accuracy: 63.33%

Epoch 8: Train Loss: 0.8790, Train Acc: 68.76%
Epoch 8: Test Accuracy: 64.00%

Epoch 9: Train Loss: 0.8248, Train Acc: 70.66%
Epoch 9: Test Accuracy: 66.57%

Epoch 10: Train Loss: 0.7743, Train Acc: 72.52%
Epoch 10: Test Accuracy: 67.26%

Epoch 11: Train Loss: 0.7210, Train Acc: 74.52%
Epoch 11: Test Accuracy: 67.59%

Epoch 12: Train Loss: 0.6717, Train Acc: 76.25%
Epoch 12: Test Accuracy: 68.31%

Epoch 13: Train Loss: 0.6197, Train Acc: 78.20

In [17]:
import torch.nn as nn
import torch.nn.functional as F

class BasicBlock(nn.Module):
    def __init__(self, in_planes, planes, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_planes, planes, kernel_size=3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)
        
        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != planes:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_planes, planes, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(planes)
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        return F.relu(out)

class ResNetSmall(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.in_planes = 24 # Changed from 16 to 24
        self.conv1 = nn.Conv2d(3, 24, kernel_size=3, stride=1, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(24)
        self.layer1 = self._make_layer(24, 2, stride=1)   # Was 16
        self.layer2 = self._make_layer(48, 2, stride=2)   # Was 32
        self.layer3 = self._make_layer(96, 2, stride=2)   # Was 64
        self.layer4 = self._make_layer(192, 2, stride=2)  # Was 128
        self.linear = nn.Linear(192, num_classes)         # Was 128

    def _make_layer(self, planes, num_blocks, stride):
        strides = [stride] + [1]*(num_blocks-1)
        layers = []
        for stride in strides:
            layers.append(BasicBlock(self.in_planes, planes, stride))
            self.in_planes = planes
        return nn.Sequential(*layers)

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.layer4(out)
        out = F.avg_pool2d(out, 4)
        out = out.view(out.size(0), -1)
        return self.linear(out)

cnn_model = ResNetSmall().to(device)
cnn_params = sum(p.numel() for p in cnn_model.parameters() if p.requires_grad)
print(f"CNN Parameters: {cnn_params / 1e6:.2f}M")

CNN Parameters: 1.58M


In [18]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(cnn_model.parameters(), lr=3e-4, weight_decay=0.05)
scheduler = CosineAnnealingLR(optimizer, T_max=20)

def train_cnn_epoch(epoch):
    cnn_model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    for inputs, labels in trainloader:
        inputs, labels = inputs.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = cnn_model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        
    train_acc = 100. * correct / total
    print(f"Epoch {epoch}: Train Loss: {running_loss/len(trainloader):.4f}, Train Acc: {train_acc:.2f}%")

def evaluate_cnn():
    cnn_model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, labels in testloader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = cnn_model(inputs)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
    return 100. * correct / total

# --- Run CNN Training ---
print("Training CNN...")
for epoch in range(1, 21):
    train_cnn_epoch(epoch)
    test_acc = evaluate_cnn()
    scheduler.step()
    print(f"Epoch {epoch}: Test Accuracy: {test_acc:.2f}%\n")

print("Finished CNN Training!")
torch.save(cnn_model.state_dict(), '/kaggle/working/cnn_cifar10.pth')
print("CNN Model saved to /kaggle/working/cnn_cifar10.pth")

Training CNN...
Epoch 1: Train Loss: 1.2823, Train Acc: 53.45%
Epoch 1: Test Accuracy: 60.29%

Epoch 2: Train Loss: 0.8291, Train Acc: 70.99%
Epoch 2: Test Accuracy: 65.15%

Epoch 3: Train Loss: 0.6301, Train Acc: 77.98%
Epoch 3: Test Accuracy: 73.09%

Epoch 4: Train Loss: 0.4917, Train Acc: 82.99%
Epoch 4: Test Accuracy: 76.47%

Epoch 5: Train Loss: 0.3766, Train Acc: 87.08%
Epoch 5: Test Accuracy: 78.44%

Epoch 6: Train Loss: 0.2702, Train Acc: 90.94%
Epoch 6: Test Accuracy: 77.18%

Epoch 7: Train Loss: 0.1810, Train Acc: 93.85%
Epoch 7: Test Accuracy: 77.82%

Epoch 8: Train Loss: 0.1193, Train Acc: 96.16%
Epoch 8: Test Accuracy: 78.69%

Epoch 9: Train Loss: 0.0702, Train Acc: 97.78%
Epoch 9: Test Accuracy: 78.07%

Epoch 10: Train Loss: 0.0501, Train Acc: 98.45%
Epoch 10: Test Accuracy: 79.12%

Epoch 11: Train Loss: 0.0330, Train Acc: 99.11%
Epoch 11: Test Accuracy: 79.70%

Epoch 12: Train Loss: 0.0204, Train Acc: 99.48%
Epoch 12: Test Accuracy: 78.76%

Epoch 13: Train Loss: 0.0105, 

In [ ]:
from torch.utils.data import Subset
import numpy as np

# 1. Create 10% subset
indices = list(range(len(trainset)))
np.random.shuffle(indices)
split = int(np.floor(0.1 * len(trainset)))
subset_indices = indices[:split]
subset_dataset = Subset(trainset, subset_indices)
subset_loader = DataLoader(subset_dataset, batch_size=64, shuffle=True, num_workers=2)

print(f"10% subset size: {len(subset_dataset)}")

# 2. Train a fresh CNN on this 10% subset for 5 epochs
cnn_subset_model = ResNetSmall().to(device)
optimizer = optim.AdamW(cnn_subset_model.parameters(), lr=3e-4, weight_decay=0.05)
criterion = nn.CrossEntropyLoss()

print("Training CNN on 10% data for 5 epochs...")
for epoch in range(1, 6):
    cnn_subset_model.train()
    for inputs, labels in subset_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = cnn_subset_model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
    
    # Evaluate on full test set
    cnn_subset_model.eval()
    val_correct, val_total = 0, 0
    with torch.no_grad():
        for inputs, labels in testloader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = cnn_subset_model(inputs)
            _, predicted = outputs.max(1)
            val_total += labels.size(0)
            val_correct += predicted.eq(labels).sum().item()
            
    print(f"Epoch {epoch}: 10% Data Test Accuracy: {100. * val_correct / val_total:.2f}%")

10% subset size: 5000
Training CNN on 10% data for 5 epochs...


In [ ]:
import matplotlib.pyplot as plt

# --- FILL IN YOUR ACTUAL NUMBERS HERE ---
vit_acc = 74.50        # REPLACE with your ViT final test accuracy
cnn_acc = 81.24        # Your CNN final test accuracy
cnn_10_acc = 55.10     # REPLACE with your 10% data result

models = ['ViT\n(1.53M)', 'CNN\n(1.58M)', 'CNN 10% Data\n(1.58M)']
accuracies = [vit_acc, cnn_acc, cnn_10_acc]
colors = ['#4C72B0', '#DD8452', '#DD8452'] # Blue for ViT, Orange for CNN

plt.figure(figsize=(9, 5))
bars = plt.bar(models, accuracies, color=colors, alpha=0.85)

for bar in bars:
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2, yval + 1, f"{yval:.2f}%", ha='center', va='bottom', fontweight='bold')

plt.title('CIFAR-10: ViT vs CNN (Fair ~1.5M Parameter Comparison)', fontsize=13)
plt.ylabel('Test Accuracy (%)', fontsize=11)
plt.ylim(0, 100)
plt.grid(axis='y', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.savefig('/kaggle/working/final_comparison.png', dpi=300)
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import seaborn as sns

# CIFAR-10 class labels
class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck']

# ---- Helper: Get predictions from a model ----
def get_predictions(model, loader, device):
    model.eval()
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, predicted = outputs.max(1)
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    return np.array(all_labels), np.array(all_preds)

# ---- Get predictions from both models ----
print("Getting ViT predictions...")
y_true_vit, y_pred_vit = get_predictions(model, testloader, device)

print("Getting CNN predictions...")
y_true_cnn, y_pred_cnn = get_predictions(cnn_model, testloader, device)

# ---- Compute confusion matrices ----
cm_vit = confusion_matrix(y_true_vit, y_pred_vit)
cm_cnn = confusion_matrix(y_true_cnn, y_pred_cnn)

print(f"\nViT Accuracy: {100 * np.trace(cm_vit) / np.sum(cm_vit):.2f}%")
print(f"CNN Accuracy: {100 * np.trace(cm_cnn) / np.sum(cm_cnn):.2f}%")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(20, 8))

# ---- ViT Confusion Matrix ----
sns.heatmap(cm_vit, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names,
            ax=axes[0], cbar_kws={'label': 'Count'})
axes[0].set_title(f'ViT Confusion Matrix (Accuracy: 69.75%)', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Predicted Label', fontsize=11)
axes[0].set_ylabel('True Label', fontsize=11)
plt.setp(axes[0].get_xticklabels(), rotation=45, ha='right')

# ---- CNN Confusion Matrix ----
sns.heatmap(cm_cnn, annot=True, fmt='d', cmap='Oranges',
            xticklabels=class_names, yticklabels=class_names,
            ax=axes[1], cbar_kws={'label': 'Count'})
axes[1].set_title(f'CNN Confusion Matrix (Accuracy: 81.24%)', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Predicted Label', fontsize=11)
axes[1].set_ylabel('True Label', fontsize=11)
plt.setp(axes[1].get_xticklabels(), rotation=45, ha='right')

plt.tight_layout()
plt.savefig('/kaggle/working/confusion_matrices.png', dpi=300, bbox_inches='tight')
plt.show()
print("Confusion matrices saved to /kaggle/working/confusion_matrices.png")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(20, 8))

# ---- ViT Normalized ----
cm_vit_norm = cm_vit.astype('float') / cm_vit.sum(axis=1, keepdims=True)
sns.heatmap(cm_vit_norm, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names,
            ax=axes[0], cbar_kws={'label': 'Proportion'}, vmin=0, vmax=1)
axes[0].set_title('ViT Normalized Confusion Matrix', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Predicted Label', fontsize=11)
axes[0].set_ylabel('True Label', fontsize=11)
plt.setp(axes[0].get_xticklabels(), rotation=45, ha='right')

# ---- CNN Normalized ----
cm_cnn_norm = cm_cnn.astype('float') / cm_cnn.sum(axis=1, keepdims=True)
sns.heatmap(cm_cnn_norm, annot=True, fmt='.2f', cmap='Oranges',
            xticklabels=class_names, yticklabels=class_names,
            ax=axes[1], cbar_kws={'label': 'Proportion'}, vmin=0, vmax=1)
axes[1].set_title('CNN Normalized Confusion Matrix', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Predicted Label', fontsize=11)
axes[1].set_ylabel('True Label', fontsize=11)
plt.setp(axes[1].get_xticklabels(), rotation=45, ha='right')

plt.tight_layout()
plt.savefig('/kaggle/working/confusion_matrices_normalized.png', dpi=300, bbox_inches='tight')
plt.show()
print("Normalized confusion matrices saved.")

In [ ]:
from sklearn.metrics import classification_report

print("=" * 60)
print("ViT Classification Report")
print("=" * 60)
print(classification_report(y_true_vit, y_pred_vit, target_names=class_names))

print("\n" + "=" * 60)
print("CNN Classification Report")
print("=" * 60)
print(classification_report(y_true_cnn, y_pred_cnn, target_names=class_names))